# CHoRUS EHR Extract

Exports the [OMOP CDM](https://ohdsi.github.io/CommonDataModel/) tables from the
CHoRUS enclave Postgres database to Parquet.

Needs `pip install -r requirements.txt` and a `.env` file holding
`POSTGRES_USER`, `POSTGRES_PASSWORD` and `POSTGRES_HOST`. See `README.md`.

Run Configuration, then whichever section below it you need.

## Configuration

In [ ]:
import logging
import os
import pathlib

import dotenv
import pandas as pd
import sqlalchemy as sa

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    force=True,  # so re-running does not stack handlers
)
log = logging.getLogger("ehr-dump")

In [ ]:
SCHEMA = "omopcdmv2"
OUTDIR = pathlib.Path("/code/Users/rocrate/omop")

# Rows per Parquet batch. Lower if a wide table exhausts memory.
CHUNKSIZE = 1_000_000

# Rows per table for the TSV samples.
SAMPLE_ROWS = 10_000

In [ ]:
dotenv.load_dotenv()


def require_env(name: str) -> str:
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"{name} is not set; see README.md")
    return value


# URL.create() encodes the password, so "@" or ":" in it is safe.
DATABASE_URL = sa.URL.create(
    "postgresql+psycopg",
    username=require_env("POSTGRES_USER"),
    password=require_env("POSTGRES_PASSWORD"),
    host=require_env("POSTGRES_HOST"),
    port=int(os.environ.get("POSTGRES_PORT", 5432)),
    database=os.environ.get("POSTGRES_DB") or None,
)

engine = sa.create_engine(DATABASE_URL)

log.info("connecting to %s", DATABASE_URL.render_as_string())  # masks password
with engine.connect() as conn:
    log.info("server version: %s", conn.exec_driver_sql("select version()").scalar())

## Query helpers

`read_sql` returns a DataFrame; `stream_sql` returns an iterator of them, for
tables larger than memory.

In [ ]:
def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(sa.text(query), conn, params=params or {})


def stream_sql(query: str, params=None, chunksize=CHUNKSIZE):
    with engine.connect().execution_options(stream_results=True) as conn:
        yield from pd.read_sql(
            sa.text(query), conn, params=params or {}, chunksize=chunksize
        )


def qualified_name(schema: str, table: str) -> str:
    """Schema-qualified identifier. Identifiers can't be bound parameters, so
    anything interpolated into SQL is checked here first."""
    for part in (schema, table):
        if not part.replace("_", "").isalnum():
            raise ValueError(f"unsafe identifier: {part!r}")
    return f"{schema}.{table}"

## Table inventory

Tables in `SCHEMA` with size and row count. Row counts are autovacuum's
`n_live_tup` estimate, which is instant where `count(*)` on the large tables is
not.

In [ ]:
def list_tables(schema: str = SCHEMA) -> pd.DataFrame:
    query = """
        select
            t.table_name,
            pg_table_size(format('%I.%I', t.table_schema, t.table_name)) as size_bytes,
            pg_size_pretty(
                pg_table_size(format('%I.%I', t.table_schema, t.table_name))
            ) as size,
            s.n_live_tup as est_rows
        from information_schema.tables t
        left join pg_stat_user_tables s
            on s.schemaname = t.table_schema
            and s.relname = t.table_name
        where t.table_schema = :schema
            and t.table_type = 'BASE TABLE'
        order by size_bytes desc
    """
    return read_sql(query, {"schema": schema})


df_tables = list_tables()
if df_tables.empty:
    raise RuntimeError(f"no tables in schema {SCHEMA!r}")

log.info(
    "%d tables, %.2f GiB, ~%s rows",
    len(df_tables),
    df_tables["size_bytes"].sum() / 2**30,
    f"{int(df_tables['est_rows'].fillna(0).sum()):,}",
)
df_tables

## Dump to Parquet

Writes every table in `SCHEMA` to `OUTDIR` as zstd Parquet, splitting tables
over `CHUNKSIZE` rows into numbered parts. Already-exported tables are skipped,
so an interrupted run resumes on re-run; pass `overwrite=True` to force.

In [ ]:
def dump_table(table: str, schema: str = SCHEMA, outdir: pathlib.Path = OUTDIR,
               overwrite: bool = False) -> list[pathlib.Path]:
    outdir = pathlib.Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    existing = sorted(outdir.glob(f"{table}.*.parquet"))
    if existing and not overwrite:
        log.info("skipping %s (%d part(s) already written)", table, len(existing))
        return existing

    written = []
    for part, df in enumerate(stream_sql(f"select * from {qualified_name(schema, table)}")):
        fp = outdir / f"{table}.{part:05d}.parquet"
        log.info("writing %s (%d rows)", fp.name, len(df))
        df.to_parquet(fp, compression="zstd", index=False)
        written.append(fp)

    if not written:
        # An empty table still needs a file, or it drops out of the crate.
        fp = outdir / f"{table}.00000.parquet"
        log.info("writing %s (empty)", fp.name)
        read_sql(
            f"select * from {qualified_name(schema, table)} where false"
        ).to_parquet(fp, compression="zstd", index=False)
        written.append(fp)

    for stale in sorted(set(existing) - set(written)):
        log.warning("removing stale part %s", stale.name)
        stale.unlink()

    return written

In [ ]:
manifest = []
for table in df_tables["table_name"]:
    log.info("exporting %s.%s", SCHEMA, table)
    for fp in dump_table(table):
        manifest.append({"table": table, "file": fp.name, "bytes": fp.stat().st_size})

df_manifest = pd.DataFrame(manifest)
log.info(
    "done: %d file(s), %.2f GiB in %s",
    len(df_manifest),
    df_manifest["bytes"].sum() / 2**30,
    OUTDIR,
)
df_manifest

## TSV samples

Optional. First `SAMPLE_ROWS` rows of each table, written to `OUTDIR/samples/`
so they aren't mistaken for the dump above. Whatever Postgres returns first, not
a random sample.

In [ ]:
sampledir = OUTDIR / "samples"
sampledir.mkdir(parents=True, exist_ok=True)

for table in df_tables["table_name"]:
    df = read_sql(
        f"select * from {qualified_name(SCHEMA, table)} limit :n",
        {"n": SAMPLE_ROWS},
    )
    fp = sampledir / f"{table}.tsv"
    log.info("writing %s (%d rows)", fp.name, len(df))
    df.to_csv(fp, sep="\t", index=False)

log.info("done: %s", sampledir)